# Session 2B — RAG-Powered Agent (Semantic Memory)

**GOAL:** Give the agent knowledge it was NEVER trained on.  
We ingest `user_preferences.txt` into a vector DB so the agent can look up YOUR personal rules before acting.

Example: If your preferences say "No meetings before 10 AM" and an email requests 9 AM, the agent will **REFUSE** and suggest an alternative.

In [1]:
import os, sys, warnings, logging

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langgraph.prebuilt import create_react_agent
from utils.tools import fetch_emails, schedule_meeting

# Paths — note: this notebook lives in session_2_frameworks/
PREFS_FILE = os.path.join(os.path.abspath('.'), 'user_preferences.txt')
CHROMA_DIR = os.path.join(os.path.abspath('.'), '.chroma_db')

print('Setup complete.')
print(f'Preferences file: {PREFS_FILE}')
print(f'ChromaDB dir: {CHROMA_DIR}')

Setup complete.
Preferences file: B:\Agentic_AI_Demos\session_2_frameworks\user_preferences.txt
ChromaDB dir: B:\Agentic_AI_Demos\session_2_frameworks\.chroma_db


### Step 1: Build the Vector Store (RAG Pipeline)

**Load → Chunk → Embed → Store**

In [2]:
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

if os.path.exists(CHROMA_DIR):
    print('Loading existing ChromaDB ...')
    vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
else:
    print('Building vector store from user_preferences.txt ...')
    loader = TextLoader(PREFS_FILE, encoding='utf-8')
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100, separators=['\n\n', '\n', '. ', ' '])
    chunks = splitter.split_documents(documents)
    print(f'   Split into {len(chunks)} chunks')
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=CHROMA_DIR)
    print(f'   Stored in ChromaDB at {CHROMA_DIR}')

print('Vector store ready.')

Building vector store from user_preferences.txt ...
   Split into 5 chunks
   Stored in ChromaDB at B:\Agentic_AI_Demos\session_2_frameworks\.chroma_db
Vector store ready.


### Step 2: Define the RAG Tool

In [3]:
@tool
def search_user_preferences(query: str) -> str:
    """Search the user's personal preferences and rules.

    Use this tool BEFORE scheduling meetings or drafting replies to check
    if the user has any relevant rules.

    Args:
        query: A natural language question about the user's preferences.
    """
    docs = vectorstore.similarity_search(query, k=3)
    if not docs:
        return 'No relevant preferences found for this query.'
    result = 'USER PREFERENCES FOUND:\n'
    for i, doc in enumerate(docs, 1):
        result += f'\n--- Rule Set {i} ---\n'
        result += doc.page_content.strip()
        result += '\n'
    return result

print('RAG tool defined.')

RAG tool defined.


### Step 3: Run the RAG-Powered Agent

In [4]:
from utils.llm_router import get_routed_llm

llm = get_routed_llm(role='worker_model')
tools = [fetch_emails, search_user_preferences, schedule_meeting]

system_msg = SystemMessage(content=(
    'You are an intelligent email assistant with access to the user\'s '
    'personal preferences database. '
    'ALWAYS search the user\'s preferences BEFORE scheduling any meeting '
    'or drafting any reply. If a requested action violates a preference, '
    'you MUST refuse and explain why, then suggest an alternative.'
))

agent = create_react_agent(llm, tools)

print('Running RAG-powered agent ...\n')

result = agent.invoke({
    'messages': [
        system_msg,
        HumanMessage(content=(
            'Fetch my recent emails. For any meeting requests found, '
            'check my personal preferences first, then schedule the '
            'meetings ONLY if they comply with my rules. '
            'If they don\'t comply, explain the conflict and suggest '
            'an alternative. Give me a complete summary at the end.'
        )),
    ]
})

print('FULL AGENT CONVERSATION:')
print('-' * 50)
for msg in result['messages']:
    role = msg.type.upper()
    content = getattr(msg, 'content', '')
    if isinstance(content, list):
        content = '\n'.join(c.get('text', '') if isinstance(c, dict) and 'text' in c else str(c) for c in content)
    if content:
        print(f'\n[{role}]:')
        print(f'  {content}')
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f'  Tool Call: {tc["name"]}({tc["args"]})')

print('\nKEY TAKEAWAY:')
print('   The agent now has SEMANTIC MEMORY — knowledge it was never trained on.')
print('   RAG Pipeline: Load -> Chunk -> Embed -> Store -> Retrieve')

Running RAG-powered agent ...

[gmail] fetched 10 emails
FULL AGENT CONVERSATION:
--------------------------------------------------

[SYSTEM]:
  You are an intelligent email assistant with access to the user's personal preferences database. ALWAYS search the user's preferences BEFORE scheduling any meeting or drafting any reply. If a requested action violates a preference, you MUST refuse and explain why, then suggest an alternative.

[HUMAN]:
  Fetch my recent emails. For any meeting requests found, check my personal preferences first, then schedule the meetings ONLY if they comply with my rules. If they don't comply, explain the conflict and suggest an alternative. Give me a complete summary at the end.
  Tool Call: fetch_emails({'limit': 10})
  Tool Call: search_user_preferences({'query': 'What are my rules for meeting scheduling?'})

[TOOL]:
  --- Email 1 ---
Date: Thu, 23 Apr 2026 19:27:53 +0530
From: Akshat Dodwad <akshatdodwad@gmail.com>
Subject: Meeting on 25th April 2026 at 1